* Reads the original datasets
* Apply augmentations
* Stores Datasets

In [1]:
import os
from pathlib import Path
import torch
import math
import torchvision
from torchvision.transforms import v2
from torchvision import tv_tensors
from sklearn.model_selection import train_test_split
from PIL import Image

In [2]:
# CONSTS
in_datasets = [("./Datasets/Original/Scenario_1", [0.8, 0.2, 0])]
out_datasets = "./Datasets/YOLO_Datasets"

AUGMENTATION_MULTIPLIER = 5 # Number of augmented clones per training image
AUG_PROBABILITY = 0.5

# ==========================================
# Torchvision v2 Augmentation Pipeline
# ==========================================
# Bounding-box aware transforms automatically modify boxes array passed alongside images
train_transform = v2.Compose([
    # 1. Flip: Horizontal
    v2.RandomHorizontalFlip(p=AUG_PROBABILITY),
    
    # 2. Rotation & Zoom: Between -10° and +10° 
    # Scaled to 1.12x to pull the canvas boundaries outward and eliminate black corners
    v2.RandomAffine(
        degrees=(-10, 10), 
        scale=(1.2, 1.2),
        interpolation=v2.InterpolationMode.BILINEAR
    ),
    
    # 3. Color & Exposure Adjustments
    # Brightness (25%), Contrast/Exposure (15%), Saturation (25%), Hue (15° -> 15/360 ≈ 0.041)
    v2.ColorJitter(
        brightness=0.25, 
        contrast=0.15, 
        saturation=0.25, 
        hue=0.041
    ),
    
    # 4. Blur: Up to 2px
    v2.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    
    # 5. Noise: Camera grain applied based on dataset probability
    v2.RandomApply([
        v2.GaussianNoise(mean=0.0, sigma=0.04, clip=True)
    ], p=AUG_PROBABILITY)
])

In [3]:
# ==========================================
# Helper Functions for YOLO Format Mapping
# ==========================================

def load_yolo_labels(label_path, img_w, img_h):
    """
    Reads YOLO labels in both formats:
      - Detection:    class cx cy w h                         (5 parts)
      - Segmentation: class x1 y1 x2 y2 ... xN yN            (odd total parts, >=7)
    
    Segmentation polygons are converted to their CXCYWH bounding box envelope.
    All coordinates are normalized [0,1] in the file and returned as absolute pixels.
    """
    boxes  = []
    labels = []

    if not os.path.exists(label_path):
        return None, None

    with open(label_path, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            try:
                class_id = int(parts[0])
                values   = [float(x) for x in parts[1:]]
            except ValueError:
                print(f"[WARN] {label_path}:{line_num} — could not parse: {line}")
                continue

            n = len(values)

            if n == 4:
                # Detection format: cx cy w h
                cx, cy, w, h = values
                boxes.append([cx * img_w, cy * img_h, w * img_w, h * img_h])

            elif n >= 6 and n % 2 == 0:
                # Segmentation format: even number of xy pairs
                coords = torch.tensor(values, dtype=torch.float32).reshape(-1, 2)
                xs = coords[:, 0] * img_w
                ys = coords[:, 1] * img_h

                x_min, x_max = xs.min().item(), xs.max().item()
                y_min, y_max = ys.min().item(), ys.max().item()

                cx = (x_min + x_max) / 2
                cy = (y_min + y_max) / 2
                w  =  x_max - x_min
                h  =  y_max - y_min
                boxes.append([cx, cy, w, h])

            elif n >= 6 and n % 2 != 0:
                # Odd count — trailing repeated closing point, drop last value
                print(f"[INFO] {label_path}:{line_num} — odd polygon ({n} values), dropping last point")
                values = values[:-1]
                coords = torch.tensor(values, dtype=torch.float32).reshape(-1, 2)
                xs = coords[:, 0] * img_w
                ys = coords[:, 1] * img_h

                x_min, x_max = xs.min().item(), xs.max().item()
                y_min, y_max = ys.min().item(), ys.max().item()

                cx = (x_min + x_max) / 2
                cy = (y_min + y_max) / 2
                w  =  x_max - x_min
                h  =  y_max - y_min
                boxes.append([cx, cy, w, h])

            else:
                print(f"[WARN] {label_path}:{line_num} — unexpected format ({n} values), skipping")
                continue

            labels.append(class_id)

    if not boxes:
        return None, None

    return (
        torch.tensor(boxes,  dtype=torch.float32),
        torch.tensor(labels, dtype=torch.int64),
    )


def save_to_yolo(img_tensor, boxes, labels, out_img_path, out_lbl_path):
    """
    Saves a torchvision image tensor as a file and writes normalized YOLO
    detection labels (CXCYWH) to the paired text file.
    """
    img = v2.functional.to_pil_image(img_tensor)
    img.save(out_img_path)

    img_w, img_h = img.size

    with open(out_lbl_path, 'w') as f:
        if boxes is not None and len(boxes) > 0:
            for box, label in zip(boxes, labels):
                cx = box[0].item() / img_w
                cy = box[1].item() / img_h
                w  = box[2].item() / img_w
                h  = box[3].item() / img_h
                f.write(f"{label.item()} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")


def save_metadata_file(input_dir):
    output_path = Path(os.path.join(out_datasets, Path(input_dir).name))

    # 4. Generate YOLO-compliant YAML File
    yaml_content = f"""# Path configuration
path: {output_path}  # dataset root dir
train: train/images          # train images (relative to 'path')
val: val/images              # val images (relative to 'path')
test: test/images            # test images (relative to 'path'), Optional

nc: 1
names: ['sapatilha']
"""
    
    with open(output_path / "dataset.yaml", "w") as f:
        f.write(yaml_content.strip())
        
    print(f"YOLO Dataset ready at: {output_path}")

In [4]:
def safe_crop_to_fill(image, boxes, target_size=640, margin=10):
    """
    Resizes an image to completely fill a target_size x target_size canvas (crop-to-fill),
    keeping all bounding boxes in view where mathematically possible.
    """
    w, h = v2.functional.get_size(image)  # returns (H, W) — careful, we need width/height

    # get_size returns [H, W] in torchvision
    orig_h, orig_w = v2.functional.get_size(image)

    # 1. Scale so the SHORT side == target_size (guarantees full coverage, no black bars)
    scale = target_size / min(orig_w, orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)

    resized_img = v2.functional.resize(image, (new_h, new_w))

    # --- No boxes: center crop ---
    if boxes is None or len(boxes) == 0:
        top  = (new_h - target_size) // 2
        left = (new_w - target_size) // 2
        cropped_img = v2.functional.crop(resized_img, top, left, target_size, target_size)
        return cropped_img, boxes

    # 2. Scale boxes from original canvas → resized canvas
    #    Work in XYXY for spatial reasoning; convert once up front.
    boxes_xyxy = tv_tensors.BoundingBoxes(
        v2.functional.convert_bounding_box_format(
            boxes,
            old_format=tv_tensors.BoundingBoxFormat.CXCYWH,
            new_format=tv_tensors.BoundingBoxFormat.XYXY,
        ),
        format="XYXY",
        canvas_size=(orig_h, orig_w),
    )
    boxes_xyxy = v2.functional.resize(boxes_xyxy, (new_h, new_w))

    # 3. Collective bounding envelope of all boxes
    x_min = torch.min(boxes_xyxy[:, 0]).item() - margin
    y_min = torch.min(boxes_xyxy[:, 1]).item() - margin
    x_max = torch.max(boxes_xyxy[:, 2]).item() + margin
    y_max = torch.max(boxes_xyxy[:, 3]).item() + margin

    def safe_offset(obj_min, obj_max, canvas_len):
        """
        Return the best top/left so the crop window [offset, offset+target_size]
        contains [obj_min, obj_max] as much as possible.
        """
        # Latest the window can start and still contain obj_min
        max_offset = int(obj_min)
        # Earliest the window can start and still contain obj_max
        min_offset = int(obj_max) - target_size
        # Clamp both to valid canvas range
        min_offset = max(0, min_offset)
        max_offset = min(canvas_len - target_size, max_offset)

        if min_offset <= max_offset:
            # There's a valid range — pick the center of it
            return (min_offset + max_offset) // 2
        else:
            # Objects wider/taller than target_size: center on them, minimize clipping
            obj_center = int((obj_min + obj_max) / 2)
            offset = obj_center - target_size // 2
            return max(0, min(canvas_len - target_size, offset))

    # 4. Compute crop origin
    if new_w > target_size:          # landscape → crop horizontally
        top  = 0
        left = safe_offset(x_min, x_max, new_w)
    elif new_h > target_size:        # portrait → crop vertically
        left = 0
        top  = safe_offset(y_min, y_max, new_h)
    else:                            # already square
        top, left = 0, 0

    # 5. Crop image
    cropped_img = v2.functional.crop(resized_img, top, left, target_size, target_size)

    # 6. Crop boxes (torchvision clips coords to new canvas automatically)
    cropped_boxes_xyxy = v2.functional.crop(boxes_xyxy, top, left, target_size, target_size)

    # 7. Convert back to CXCYWH for the caller
    cropped_boxes = tv_tensors.BoundingBoxes(
        v2.functional.convert_bounding_box_format(
            cropped_boxes_xyxy,
            new_format=tv_tensors.BoundingBoxFormat.CXCYWH,
        ),
        format="CXCYWH",
        canvas_size=(target_size, target_size),
    )

    return cropped_img, cropped_boxes

In [5]:
def IngestDataset(input_dir, split_ratios):
    input_img_dir = os.path.join(input_dir, 'images')
    input_lbl_dir = os.path.join(input_dir, 'labels')
    output_dir = os.path.join(out_datasets, Path(input_dir).name)

    # Establish directories
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
        os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)

    all_images = [f for f in os.listdir(input_img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    # Stratify split
    train_imgs, val_imgs = train_test_split(all_images, test_size=(1 - split_ratios[0]), random_state=42)
    val_test_ratio = split_ratios[2] / (split_ratios[1] + split_ratios[2])

    if split_ratios[2] > 0:
        val_imgs, test_imgs = train_test_split(val_imgs, test_size=val_test_ratio, random_state=42)
    else:
        test_imgs = []  # Cleaned: Changed from {} to [] for type consistency

    splits = {'train': (train_imgs, True), 'val': (val_imgs, False), 'test': (test_imgs, False)}
    
    for split_name, (img_list, is_train) in splits.items():
        print(f"Processing {split_name} split...")
        multiplier = AUGMENTATION_MULTIPLIER if is_train else 1
        
        for img_name in img_list:
            img_path = os.path.join(input_img_dir, img_name)
            lbl_path = os.path.join(input_lbl_dir, os.path.splitext(img_name)[0] + '.txt')
            
            # Load original image using PIL
            try:
                orig_img = Image.open(img_path).convert("RGB")
            except Exception:
                continue
                
            orig_w, orig_h = orig_img.size
            boxes, labels = load_yolo_labels(lbl_path, orig_w, orig_h)

            # 1. Generate Clean Ground-Truth Base Copy
            crp_og_img, crp_og_boxes = safe_crop_to_fill(orig_img, boxes, target_size=640)
            crp_img_tensor = v2.functional.to_image(crp_og_img)

            # Save Base Copy
                   # Save clean base copy
            out_base = os.path.splitext(img_name)[0]
            save_to_yolo(
                crp_img_tensor, crp_og_boxes, labels,
                os.path.join(output_dir, split_name, 'images', out_base + '.jpg'),
                os.path.join(output_dir, split_name, 'labels', out_base + '.txt'),
            )     
           
            if is_train:
                for i in range(AUGMENTATION_MULTIPLIER):
                    img_t = crp_img_tensor.clone()

                    # BUG FIX 1: use crp_og_boxes (640×640 space), not original boxes
                    if crp_og_boxes is not None and len(crp_og_boxes) > 0:
                        box_t = tv_tensors.BoundingBoxes(
                            crp_og_boxes.clone(),
                            format="CXCYWH",
                            canvas_size=(640, 640),
                        )
                        # BUG FIX 2: pass both image AND boxes through the transform together
                        img_t, box_t = train_transform(img_t, box_t)
                    else:
                        img_t = train_transform(img_t)
                        box_t = crp_og_boxes

                    out_name = out_base + f"_aug_{i}"
                    save_to_yolo(
                        img_t, box_t, labels,
                        os.path.join(output_dir, split_name, 'images', out_name + '.jpg'),
                        os.path.join(output_dir, split_name, 'labels', out_name + '.txt'),
                    )

    print(f"Dataset processing complete! Stored in: {output_dir}")

In [39]:
for path, split_config in in_datasets:
    IngestDataset(path, split_config)
    save_metadata_file(path)

Processing train split...
Processing val split...
Processing test split...
Dataset processing complete! Stored in: ./Datasets/YOLO_Datasets\Scenario_1
YOLO Dataset ready at: Datasets\YOLO_Datasets\Scenario_1


In [6]:
in_datasets = [("./Datasets/Original/Cafe_1", [0.8, 0.2, 0])]
out_datasets = "./Datasets/YOLO_Datasets"

for path, split_config in in_datasets:
    IngestDataset(path, split_config)
    save_metadata_file(path)

Processing train split...
Processing val split...
Processing test split...
Dataset processing complete! Stored in: ./Datasets/YOLO_Datasets\Cafe_1
YOLO Dataset ready at: Datasets\YOLO_Datasets\Cafe_1
